In [1]:
!apt-get update -qq
!apt-get install -y flex gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison flex-doc
The following NEW packages will be installed:
  flex libfl-dev libfl2
0 upgraded, 3 newly installed, 0 to remove and 137 not upgraded.
Need to get 324 kB of archives.
After this operation, 1,148 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 [10.7 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl-dev amd64 2.6.4-8build2 [6,236 B]
Fet

In [3]:
%%writefile symtab.l
%{
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

struct symtab {
    char name[30];
    int type;
} symtab[100];

int sc = 0;

int lookup(char *s) {
    int i;
    for (i = 0; i < sc; i++)
        if (strcmp(symtab[i].name, s) == 0)
            return i;
    return -1;
}

void insert(char *s) {
    if (lookup(s) == -1) {
        strcpy(symtab[sc].name, s);
        symtab[sc].type = 1;
        sc++;
    }
}
%}

DIGIT [0-9]
ID [a-zA-Z_][a-zA-Z0-9_]*

%%

"/*"([^*]|\*+[^*/])*\*+"/" {
    printf("Comment : %s\n", yytext);
}

"//".* {
    printf("Comment : %s\n", yytext);
}

{ID} {
    insert(yytext);
    printf("Identifier : %s\n", yytext);
}

{DIGIT}+ {
    printf("Constant : %s\n", yytext);
}

"+"|"-"|"*"|"/"|"="|"<"|">" {
    printf("Operator : %s\n", yytext);
}

[ \t\n] {
    /* skip whitespace */
}

. {
    /* ignore other characters */
}

%%

int yywrap() {
    return 1;
}

int main(int argc, char *argv[]) {

    if (argc < 2) {
        printf("Usage: %s <input file>\n", argv[0]);
        return 1;
    }

    yyin = fopen(argv[1], "r");

    if (!yyin) {
        printf("Cannot open file %s\n", argv[1]);
        return 1;
    }

    yylex();

    printf("\nSYMBOL TABLE\n");
    printf("S.No\tName\n");

    int i;
    for (i = 0; i < sc; i++)
        printf("%d\t%s\n", i + 1, symtab[i].name);

    fclose(yyin);

    return 0;
}

Overwriting symtab.l


In [6]:
!flex symtab.l
!gcc lex.yy.c -o symtab

In [9]:
%%writefile input.txt
int a = 10;
float b = 20;
a = a + b;
// this is a comment

Overwriting input.txt


In [8]:
!./symtab input.txt

Identifier : int
Identifier : a
Operator : =
Constant : 10
Identifier : float
Identifier : b
Operator : =
Constant : 20
Identifier : a
Operator : =
Identifier : a
Operator : +
Identifier : b
Comment : // this is a comment

SYMBOL TABLE
S.No	Name
1	int
2	a
3	float
4	b
